# Module 5 • Neural Networks for Natural Language Processing

# Lesson 29 • Sequence-to-Sequence Learning with Encoder–Decoder Networks

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 150–190 minutes  
**Execution target:** CPU

---

## Scope

This lesson introduces sequence-to-sequence learning with recurrent
encoder–decoder networks. It develops sequence vocabularies, special tokens,
variable-length batching, recurrent encoding, autoregressive decoding,
teacher forcing, masked sequence loss, greedy inference, sequence-level
evaluation, exposure bias, and the fixed-vector bottleneck that motivates
attention.

The executable experiment trains a compact English-to-French command
translator locally with PyTorch. It does not require a GPU or external
downloads.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain sequence-to-sequence learning;
- distinguish encoder and decoder responsibilities;
- use PAD, UNK, BOS, and EOS tokens;
- construct separate source and target vocabularies;
- encode variable-length source and target sequences;
- implement a recurrent encoder;
- implement an autoregressive recurrent decoder;
- explain teacher forcing;
- calculate masked sequence cross-entropy;
- train an encoder–decoder model;
- perform greedy decoding;
- identify exposure bias;
- evaluate exact match, token accuracy, and BLEU-like scores;
- inspect decoding errors;
- explain the fixed-vector bottleneck;
- discuss Arabic sequence-to-sequence considerations.

## Table of Contents

1. What Is Sequence-to-Sequence Learning?
2. Encoder–Decoder Architecture
3. Source and Target Sequences
4. Special Tokens
5. Autoregressive Decoding
6. Teacher Forcing
7. Sequence Cross-Entropy
8. Inference Versus Training
9. Toy Translation Task
10. Dataset Construction
11. Train, Validation, and Test Splits
12. Tokenization
13. Vocabulary Construction
14. Numerical Encoding
15. Dataset and Collation
16. Encoder
17. Decoder
18. Complete Seq2Seq Model
19. Shape Inspection
20. Training Utilities
21. Teacher-Forcing Schedule
22. Training the Model
23. Learning Curves
24. Greedy Decoding
25. Qualitative Translation
26. Exact-Match Evaluation
27. Token-Level Accuracy
28. BLEU-Like Evaluation
29. Error Analysis
30. Exposure Bias
31. Decoding Length
32. Repetition and Premature EOS
33. Beam Search Concept
34. Fixed-Vector Bottleneck
35. Bidirectional Encoders
36. Multi-Layer Encoder–Decoders
37. Regularization and Stability
38. Computational Cost
39. Arabic and Multilingual Considerations
40. Reproducibility and Reporting
41. Knowledge Check
42. Exercises
43. Summary and Next Lesson

# 1. What Is Sequence-to-Sequence Learning?

Sequence-to-sequence learning maps an input sequence to an output sequence.

Examples include:

- machine translation;
- text normalization;
- transliteration;
- summarization;
- dialogue generation;
- grammatical correction.

In [ ]:
import copy
import math
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset

task_examples = pd.DataFrame(
    [
        ("Machine translation", "English sentence", "French sentence"),
        ("Transliteration", "Arabic script", "Latin script"),
        ("Summarization", "Long document", "Short summary"),
        ("Correction", "Noisy sentence", "Corrected sentence"),
    ],
    columns=["Task", "Source", "Target"],
)

task_examples

Unlike classification, the output length is not fixed and the output tokens
are dependent on one another.

# 2. Encoder–Decoder Architecture

A recurrent encoder–decoder has two main components:

```text
source tokens
     ↓
  encoder
     ↓
context state
     ↓
  decoder
     ↓
target tokens
```

The encoder compresses the source sequence. The decoder generates the target
sequence one token at a time.

In [ ]:
component_roles = pd.DataFrame(
    [
        ("Encoder", "reads source sequence", "context state"),
        ("Decoder", "generates target sequence", "token probabilities"),
        ("Output layer", "maps hidden state to target vocabulary", "logits"),
    ],
    columns=["Component", "Role", "Output"],
)

component_roles

# 3. Source and Target Sequences

Source and target languages commonly use separate vocabularies.

Example:

```text
source: open the red door
target: ouvre la porte rouge
```

The source and target may differ in token order, vocabulary size, morphology,
and sequence length.

# 4. Special Tokens

Common sequence tokens:

- `<PAD>`: batch padding;
- `<UNK>`: unknown word;
- `<BOS>`: beginning of target sequence;
- `<EOS>`: end of sequence.

In [ ]:
special_tokens = pd.DataFrame(
    [
        ("<PAD>", "padding", "ignored by loss"),
        ("<UNK>", "unknown token", "fallback vocabulary entry"),
        ("<BOS>", "beginning of sequence", "first decoder input"),
        ("<EOS>", "end of sequence", "stops decoding"),
    ],
    columns=["Token", "Purpose", "Use"],
)

special_tokens

# 5. Autoregressive Decoding

At decoding step \(t\), the model predicts:

\[
P(y_t \mid y_{<t}, x)
\]

The prediction depends on the source sequence and previously generated target
tokens.

Autoregressive generation is sequential: token \(t+1\) cannot be generated
before token \(t\).

# 6. Teacher Forcing

During training, the decoder can receive the correct previous target token
instead of its own previous prediction.

```text
training input:    correct previous token
inference input:   predicted previous token
```

In [ ]:
teacher_forcing_summary = pd.DataFrame(
    [
        ("High ratio", "faster supervised learning", "larger train–test gap"),
        ("Low ratio", "more self-generated context", "harder optimization"),
        ("Scheduled ratio", "gradual transition", "extra hyperparameter"),
    ],
    columns=["Strategy", "Benefit", "Risk"],
)

teacher_forcing_summary

# 7. Sequence Cross-Entropy

Sequence loss sums or averages token losses over valid target positions.

Padding tokens must be ignored.

In [ ]:
example_logits = torch.tensor(
    [
        [
            [2.0, 0.5, -1.0],
            [0.1, 1.8, -0.3],
        ]
    ],
    dtype=torch.float32,
)

example_targets = torch.tensor(
    [[0, 1]],
    dtype=torch.long,
)

example_loss = nn.CrossEntropyLoss()(
    example_logits.reshape(-1, 3),
    example_targets.reshape(-1),
)

print("Example sequence loss:", float(example_loss))

# 8. Inference Versus Training

Training can process known target sequences. Inference must decide:

- which token to emit;
- when to stop;
- how to recover from earlier mistakes.

In [ ]:
train_inference = pd.DataFrame(
    [
        ("Previous token", "gold or predicted", "predicted"),
        ("Target length", "known", "unknown"),
        ("Stopping", "target mask", "EOS or limit"),
        ("Search", "not required", "greedy or beam"),
    ],
    columns=["Property", "Training", "Inference"],
)

train_inference

# 9. Toy Translation Task

The notebook uses English commands and deterministic French translations.

Vocabulary dimensions vary through:

- five verbs;
- five nouns;
- six modifiers.

The task is intentionally controlled so the model can learn sequence
composition on CPU.

# 10. Dataset Construction

In [ ]:
verbs = {
    "open": "ouvre",
    "close": "ferme",
    "find": "trouve",
    "take": "prends",
    "move": "deplace",
}

nouns = {
    "door": {
        "article": "la",
        "translation": "porte",
        "gender": "f",
    },
    "window": {
        "article": "la",
        "translation": "fenetre",
        "gender": "f",
    },
    "box": {
        "article": "la",
        "translation": "boite",
        "gender": "f",
    },
    "book": {
        "article": "le",
        "translation": "livre",
        "gender": "m",
    },
    "key": {
        "article": "la",
        "translation": "cle",
        "gender": "f",
    },
}

modifiers = {
    "red": {
        "position": "after",
        "m": "rouge",
        "f": "rouge",
    },
    "blue": {
        "position": "after",
        "m": "bleu",
        "f": "bleue",
    },
    "green": {
        "position": "after",
        "m": "vert",
        "f": "verte",
    },
    "yellow": {
        "position": "after",
        "m": "jaune",
        "f": "jaune",
    },
    "small": {
        "position": "before",
        "m": "petit",
        "f": "petite",
    },
    "big": {
        "position": "before",
        "m": "grand",
        "f": "grande",
    },
}


def build_translation(
    verb: str,
    noun: str,
    modifier: str,
) -> tuple[str, str]:
    noun_info = nouns[noun]
    modifier_info = modifiers[modifier]
    gender = noun_info["gender"]

    source = f"{verb} the {modifier} {noun}"

    translated_modifier = modifier_info[gender]
    article = noun_info["article"]
    translated_noun = noun_info["translation"]

    if modifier_info["position"] == "before":
        target = (
            f"{verbs[verb]} "
            f"{article} "
            f"{translated_modifier} "
            f"{translated_noun}"
        )
    else:
        target = (
            f"{verbs[verb]} "
            f"{article} "
            f"{translated_noun} "
            f"{translated_modifier}"
        )

    return source, target


records = []

for verb in verbs:
    for noun in nouns:
        for modifier in modifiers:
            source, target = build_translation(
                verb,
                noun,
                modifier,
            )
            records.append(
                {
                    "source": source,
                    "target": target,
                    "verb": verb,
                    "noun": noun,
                    "modifier": modifier,
                }
            )

dataset = pd.DataFrame(records)

print("Sentence pairs:", len(dataset))
dataset.head()

In [ ]:
dataset.sample(
    8,
    random_state=42,
)[
    ["source", "target"]
].reset_index(drop=True)

# 11. Train, Validation, and Test Splits

The split is deterministic. Vocabulary is built from training data only.

In [ ]:
train_frame, test_frame = train_test_split(
    dataset,
    test_size=0.15,
    random_state=42,
)

train_frame, validation_frame = train_test_split(
    train_frame,
    test_size=0.1765,
    random_state=42,
)

train_frame = train_frame.reset_index(drop=True)
validation_frame = validation_frame.reset_index(drop=True)
test_frame = test_frame.reset_index(drop=True)

split_summary = pd.Series(
    {
        "training": len(train_frame),
        "validation": len(validation_frame),
        "test": len(test_frame),
    }
)

split_summary

In [ ]:
coverage_check = pd.DataFrame(
    {
        "train_verbs": sorted(train_frame["verb"].unique()),
        "expected_verbs": sorted(verbs),
    }
)

print(
    "All verbs represented:",
    set(train_frame["verb"]) == set(verbs),
)
print(
    "All nouns represented:",
    set(train_frame["noun"]) == set(nouns),
)
print(
    "All modifiers represented:",
    set(train_frame["modifier"]) == set(modifiers),
)

# 12. Tokenization

The toy corpus uses lowercase whitespace-compatible tokens.

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:[-']\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


tokenize(
    "Open the small blue window."
)

Real translation systems need language-specific normalization and
tokenization policies.

# 13. Vocabulary Construction

In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
BOS_TOKEN = "<BOS>"
EOS_TOKEN = "<EOS>"


class Vocabulary:
    def __init__(
        self,
        texts,
        include_bos: bool,
    ):
        counts = Counter(
            token
            for text in texts
            for token in tokenize(text)
        )

        special = [
            PAD_TOKEN,
            UNK_TOKEN,
        ]

        if include_bos:
            special.append(BOS_TOKEN)

        special.append(EOS_TOKEN)

        self.index_to_token = (
            special
            + sorted(counts)
        )

        self.token_to_index = {
            token: index
            for index, token in enumerate(
                self.index_to_token
            )
        }

        self.pad_id = self.token_to_index[
            PAD_TOKEN
        ]
        self.unk_id = self.token_to_index[
            UNK_TOKEN
        ]
        self.eos_id = self.token_to_index[
            EOS_TOKEN
        ]

        self.bos_id = (
            self.token_to_index[BOS_TOKEN]
            if include_bos
            else None
        )

    def __len__(self):
        return len(self.index_to_token)

    def encode(
        self,
        text: str,
        add_bos: bool = False,
    ) -> list[int]:
        ids = []

        if add_bos:
            if self.bos_id is None:
                raise ValueError(
                    "This vocabulary has no BOS token"
                )
            ids.append(self.bos_id)

        ids.extend(
            self.token_to_index.get(
                token,
                self.unk_id,
            )
            for token in tokenize(text)
        )

        ids.append(self.eos_id)

        return ids

    def decode(
        self,
        ids,
        stop_at_eos: bool = True,
    ) -> list[str]:
        tokens = []

        for token_id in ids:
            token = self.index_to_token[
                int(token_id)
            ]

            if (
                stop_at_eos
                and token == EOS_TOKEN
            ):
                break

            if token not in {
                PAD_TOKEN,
                BOS_TOKEN,
            }:
                tokens.append(token)

        return tokens


source_vocabulary = Vocabulary(
    train_frame["source"],
    include_bos=False,
)

target_vocabulary = Vocabulary(
    train_frame["target"],
    include_bos=True,
)

print("Source vocabulary:", len(source_vocabulary))
print("Target vocabulary:", len(target_vocabulary))

Separate vocabularies allow different scripts and lexical inventories.

# 14. Numerical Encoding

In [ ]:
example_source = train_frame.loc[
    0,
    "source",
]
example_target = train_frame.loc[
    0,
    "target",
]

source_ids = source_vocabulary.encode(
    example_source
)
target_ids = target_vocabulary.encode(
    example_target,
    add_bos=True,
)

print(example_source)
print(source_ids)
print()
print(example_target)
print(target_ids)
print(
    target_vocabulary.decode(
        target_ids
    )
)

The decoder input begins with BOS. EOS is the supervised stopping token.

# 15. Dataset and Collation

In [ ]:
class TranslationDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
    ):
        self.frame = frame.reset_index(
            drop=True
        )

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]

        source = torch.tensor(
            source_vocabulary.encode(
                row["source"]
            ),
            dtype=torch.long,
        )

        target = torch.tensor(
            target_vocabulary.encode(
                row["target"],
                add_bos=True,
            ),
            dtype=torch.long,
        )

        return (
            source,
            target,
            row["source"],
            row["target"],
        )


def collate_translation_batch(batch):
    sources, targets, source_texts, target_texts = zip(
        *batch
    )

    source_lengths = torch.tensor(
        [len(sequence) for sequence in sources],
        dtype=torch.long,
    )

    target_lengths = torch.tensor(
        [len(sequence) for sequence in targets],
        dtype=torch.long,
    )

    max_source_length = int(
        source_lengths.max().item()
    )
    max_target_length = int(
        target_lengths.max().item()
    )

    padded_sources = torch.full(
        (
            len(sources),
            max_source_length,
        ),
        source_vocabulary.pad_id,
        dtype=torch.long,
    )

    padded_targets = torch.full(
        (
            len(targets),
            max_target_length,
        ),
        target_vocabulary.pad_id,
        dtype=torch.long,
    )

    for row, sequence in enumerate(sources):
        padded_sources[
            row,
            :len(sequence),
        ] = sequence

    for row, sequence in enumerate(targets):
        padded_targets[
            row,
            :len(sequence),
        ] = sequence

    return {
        "source_ids": padded_sources,
        "source_lengths": source_lengths,
        "target_ids": padded_targets,
        "target_lengths": target_lengths,
        "source_texts": list(source_texts),
        "target_texts": list(target_texts),
    }

In [ ]:
train_dataset = TranslationDataset(
    train_frame
)
validation_dataset = TranslationDataset(
    validation_frame
)
test_dataset = TranslationDataset(
    test_frame
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_translation_batch,
    generator=torch.Generator().manual_seed(42),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_translation_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_translation_batch,
)

batch = next(iter(train_loader))

print("Source batch:", batch["source_ids"].shape)
print("Target batch:", batch["target_ids"].shape)

# 16. Encoder

The encoder:

1. embeds source token IDs;
2. packs valid positions;
3. runs a GRU;
4. returns the final hidden state.

In [ ]:
class Encoder(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        embedding_dim: int,
        hidden_dim: int,
        padding_id: int,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dim,
            padding_idx=padding_id,
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )

    def forward(
        self,
        source_ids: torch.Tensor,
        source_lengths: torch.Tensor,
    ) -> torch.Tensor:
        embedded = self.embedding(
            source_ids
        )

        packed = pack_padded_sequence(
            embedded,
            source_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, hidden = self.gru(packed)

        return hidden

The hidden state shape is `(layers, batch, hidden_dim)`.

# 17. Decoder

At each step, the decoder receives:

- one target token;
- the previous hidden state.

It returns target-vocabulary logits and the updated hidden state.

In [ ]:
class Decoder(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        embedding_dim: int,
        hidden_dim: int,
        padding_id: int,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dim,
            padding_idx=padding_id,
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )

        self.output_layer = nn.Linear(
            hidden_dim,
            vocabulary_size,
        )

    def forward_step(
        self,
        token_ids: torch.Tensor,
        hidden: torch.Tensor,
    ):
        embedded = self.embedding(
            token_ids
        ).unsqueeze(1)

        output, hidden = self.gru(
            embedded,
            hidden,
        )

        logits = self.output_layer(
            output.squeeze(1)
        )

        return logits, hidden

# 18. Complete Seq2Seq Model

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(
        self,
        encoder: Encoder,
        decoder: Decoder,
        target_pad_id: int,
    ):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.target_pad_id = target_pad_id

    def forward(
        self,
        source_ids: torch.Tensor,
        source_lengths: torch.Tensor,
        target_ids: torch.Tensor,
        teacher_forcing_ratio: float,
    ) -> torch.Tensor:
        batch_size = source_ids.size(0)
        target_length = target_ids.size(1)
        vocabulary_size = (
            self.decoder.output_layer.out_features
        )

        logits_all = torch.zeros(
            batch_size,
            target_length - 1,
            vocabulary_size,
            device=source_ids.device,
        )

        hidden = self.encoder(
            source_ids,
            source_lengths,
        )

        decoder_input = target_ids[
            :,
            0,
        ]

        for time_step in range(
            1,
            target_length,
        ):
            logits, hidden = (
                self.decoder.forward_step(
                    decoder_input,
                    hidden,
                )
            )

            logits_all[
                :,
                time_step - 1,
                :,
            ] = logits

            predicted = logits.argmax(
                dim=1
            )

            use_teacher = (
                random.random()
                < teacher_forcing_ratio
            )

            decoder_input = (
                target_ids[:, time_step]
                if use_teacher
                else predicted
            )

        return logits_all

In [ ]:
DEVICE = torch.device("cpu")

torch.manual_seed(42)

encoder = Encoder(
    vocabulary_size=len(
        source_vocabulary
    ),
    embedding_dim=32,
    hidden_dim=48,
    padding_id=source_vocabulary.pad_id,
)

decoder = Decoder(
    vocabulary_size=len(
        target_vocabulary
    ),
    embedding_dim=32,
    hidden_dim=48,
    padding_id=target_vocabulary.pad_id,
)

model = Seq2Seq(
    encoder,
    decoder,
    target_vocabulary.pad_id,
).to(DEVICE)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
    ),
)

# 19. Shape Inspection

In [ ]:
with torch.no_grad():
    shape_logits = model(
        batch["source_ids"].to(DEVICE),
        batch["source_lengths"].to(DEVICE),
        batch["target_ids"].to(DEVICE),
        teacher_forcing_ratio=1.0,
    )

print(
    "Source IDs:",
    batch["source_ids"].shape,
)
print(
    "Target IDs:",
    batch["target_ids"].shape,
)
print(
    "Decoder logits:",
    shape_logits.shape,
)

The model predicts every target token after BOS, including EOS.

# 20. Training Utilities

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


loss_function = nn.CrossEntropyLoss(
    ignore_index=target_vocabulary.pad_id
)


def sequence_loss(
    logits: torch.Tensor,
    target_ids: torch.Tensor,
) -> torch.Tensor:
    expected = target_ids[:, 1:]

    return loss_function(
        logits.reshape(
            -1,
            logits.size(-1),
        ),
        expected.reshape(-1),
    )


def evaluate_loss(
    model: nn.Module,
    loader: DataLoader,
) -> float:
    model.eval()

    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for batch in loader:
            source_ids = batch[
                "source_ids"
            ].to(DEVICE)
            source_lengths = batch[
                "source_lengths"
            ].to(DEVICE)
            target_ids = batch[
                "target_ids"
            ].to(DEVICE)

            logits = model(
                source_ids,
                source_lengths,
                target_ids,
                teacher_forcing_ratio=0.0,
            )

            loss = sequence_loss(
                logits,
                target_ids,
            )

            total_loss += float(
                loss.item()
            )
            total_batches += 1

    return (
        total_loss
        / max(total_batches, 1)
    )

Validation uses no teacher forcing to better reflect free-running behavior.

# 21. Teacher-Forcing Schedule

The ratio decays during training.

In [ ]:
def teacher_forcing_schedule(
    epoch: int,
    total_epochs: int,
    start: float = 1.0,
    end: float = 0.45,
) -> float:
    fraction = (
        epoch
        / max(total_epochs - 1, 1)
    )

    return (
        start
        + fraction
        * (end - start)
    )


schedule_frame = pd.DataFrame(
    {
        "epoch": range(60),
        "ratio": [
            teacher_forcing_schedule(
                epoch,
                60,
            )
            for epoch in range(60)
        ],
    }
)

schedule_frame.iloc[
    [0, 15, 30, 45, 59]
]

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    schedule_frame["epoch"],
    schedule_frame["ratio"],
)
plt.xlabel("Epoch")
plt.ylabel("Teacher-forcing ratio")
plt.title("Teacher-Forcing Schedule")
plt.tight_layout()
plt.show()

# 22. Training the Model

In [ ]:
def train_seq2seq(
    model: nn.Module,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    epochs: int = 70,
    learning_rate: float = 0.008,
    clip_norm: float = 5.0,
    patience: int = 12,
):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )
    best_validation_loss = float(
        "inf"
    )
    epochs_without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()

        ratio = teacher_forcing_schedule(
            epoch,
            epochs,
        )

        total_loss = 0.0
        total_batches = 0
        gradient_norms = []

        for batch in train_loader:
            source_ids = batch[
                "source_ids"
            ].to(DEVICE)
            source_lengths = batch[
                "source_lengths"
            ].to(DEVICE)
            target_ids = batch[
                "target_ids"
            ].to(DEVICE)

            optimizer.zero_grad()

            logits = model(
                source_ids,
                source_lengths,
                target_ids,
                teacher_forcing_ratio=ratio,
            )

            loss = sequence_loss(
                logits,
                target_ids,
            )

            loss.backward()

            gradient_norm = clip_grad_norm_(
                model.parameters(),
                max_norm=clip_norm,
            )

            optimizer.step()

            total_loss += float(
                loss.item()
            )
            total_batches += 1
            gradient_norms.append(
                float(gradient_norm)
            )

        training_loss = (
            total_loss
            / max(total_batches, 1)
        )

        validation_loss = evaluate_loss(
            model,
            validation_loader,
        )

        history.append(
            {
                "epoch": epoch,
                "training_loss": training_loss,
                "validation_loss": validation_loss,
                "teacher_forcing_ratio": ratio,
                "mean_gradient_norm": float(
                    np.mean(
                        gradient_norms
                    )
                ),
            }
        )

        if (
            validation_loss
            < best_validation_loss
            - 1e-5
        ):
            best_validation_loss = (
                validation_loss
            )
            best_state = copy.deepcopy(
                model.state_dict()
            )
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= patience
        ):
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
    )


set_seed(42)

trained_model, training_history = train_seq2seq(
    model,
    train_loader,
    validation_loader,
)

print(
    "Epochs completed:",
    len(training_history),
)
print(
    "Best validation loss:",
    round(
        training_history[
            "validation_loss"
        ].min(),
        4,
    ),
)

# 23. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history["training_loss"],
    label="Training loss",
)
plt.plot(
    training_history["epoch"],
    training_history["validation_loss"],
    label="Validation loss",
)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Seq2Seq Learning Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history["mean_gradient_norm"],
)
plt.xlabel("Epoch")
plt.ylabel("Mean pre-clipping gradient norm")
plt.title("Encoder–Decoder Gradient Norms")
plt.tight_layout()
plt.show()

# 24. Greedy Decoding

Greedy decoding selects the highest-probability token at each step.

In [ ]:
def greedy_decode(
    model: Seq2Seq,
    source_text: str,
    maximum_length: int = 10,
):
    model.eval()

    source_tensor = torch.tensor(
        [
            source_vocabulary.encode(
                source_text
            )
        ],
        dtype=torch.long,
        device=DEVICE,
    )

    source_length = torch.tensor(
        [source_tensor.size(1)],
        dtype=torch.long,
        device=DEVICE,
    )

    with torch.no_grad():
        hidden = model.encoder(
            source_tensor,
            source_length,
        )

        decoder_input = torch.tensor(
            [target_vocabulary.bos_id],
            dtype=torch.long,
            device=DEVICE,
        )

        generated_ids = []
        step_probabilities = []

        for _ in range(maximum_length):
            logits, hidden = (
                model.decoder.forward_step(
                    decoder_input,
                    hidden,
                )
            )

            probabilities = torch.softmax(
                logits,
                dim=1,
            )

            predicted_id = int(
                probabilities.argmax(
                    dim=1
                ).item()
            )

            generated_ids.append(
                predicted_id
            )
            step_probabilities.append(
                float(
                    probabilities[
                        0,
                        predicted_id,
                    ].item()
                )
            )

            if (
                predicted_id
                == target_vocabulary.eos_id
            ):
                break

            decoder_input = torch.tensor(
                [predicted_id],
                dtype=torch.long,
                device=DEVICE,
            )

    tokens = target_vocabulary.decode(
        generated_ids
    )

    return {
        "translation": " ".join(tokens),
        "token_ids": generated_ids,
        "probabilities": step_probabilities,
    }


greedy_decode(
    trained_model,
    "open the red door",
)

Greedy search is fast but may miss a better complete sequence.

# 25. Qualitative Translation

In [ ]:
qualitative_rows = []

for row in test_frame.head(10).itertuples(
    index=False
):
    decoded = greedy_decode(
        trained_model,
        row.source,
    )

    qualitative_rows.append(
        {
            "source": row.source,
            "reference": row.target,
            "prediction": decoded[
                "translation"
            ],
        }
    )

pd.DataFrame(qualitative_rows)

# 26. Exact-Match Evaluation

Exact match requires every predicted token to match the reference.

In [ ]:
def evaluate_translations(
    model: Seq2Seq,
    frame: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for row in frame.itertuples(
        index=False
    ):
        decoded = greedy_decode(
            model,
            row.source,
        )

        prediction = decoded[
            "translation"
        ]

        rows.append(
            {
                "source": row.source,
                "reference": row.target,
                "prediction": prediction,
                "exact_match": (
                    prediction.strip()
                    == row.target.strip()
                ),
                "mean_token_confidence": (
                    float(
                        np.mean(
                            decoded[
                                "probabilities"
                            ]
                        )
                    )
                    if decoded[
                        "probabilities"
                    ]
                    else 0.0
                ),
            }
        )

    return pd.DataFrame(rows)


test_results = evaluate_translations(
    trained_model,
    test_frame,
)

print(
    "Exact-match accuracy:",
    round(
        test_results[
            "exact_match"
        ].mean(),
        3,
    ),
)

Exact match is strict and useful for controlled tasks. It may undervalue
semantically acceptable alternatives in open translation.

# 27. Token-Level Accuracy

In [ ]:
def position_token_accuracy(
    reference: str,
    prediction: str,
) -> float:
    reference_tokens = tokenize(
        reference
    )
    prediction_tokens = tokenize(
        prediction
    )

    maximum = max(
        len(reference_tokens),
        len(prediction_tokens),
        1,
    )

    matches = sum(
        left == right
        for left, right in zip(
            reference_tokens,
            prediction_tokens,
        )
    )

    return matches / maximum


test_results[
    "token_accuracy"
] = [
    position_token_accuracy(
        reference,
        prediction,
    )
    for reference, prediction in zip(
        test_results["reference"],
        test_results["prediction"],
    )
]

print(
    "Mean token accuracy:",
    round(
        test_results[
            "token_accuracy"
        ].mean(),
        3,
    ),
)

Position accuracy penalizes insertions, deletions, and shifted sequences.

# 28. BLEU-Like Evaluation

This notebook implements a compact educational sentence BLEU approximation
with add-one smoothing.

In [ ]:
def ngrams(
    tokens: list[str],
    order: int,
) -> Counter:
    return Counter(
        tuple(
            tokens[index:index + order]
        )
        for index in range(
            len(tokens) - order + 1
        )
    )


def sentence_bleu_like(
    reference: str,
    prediction: str,
    maximum_order: int = 4,
) -> float:
    reference_tokens = tokenize(reference)
    prediction_tokens = tokenize(prediction)

    if not prediction_tokens:
        return 0.0

    precisions = []

    for order in range(
        1,
        maximum_order + 1,
    ):
        predicted_ngrams = ngrams(
            prediction_tokens,
            order,
        )
        reference_ngrams = ngrams(
            reference_tokens,
            order,
        )

        clipped = sum(
            min(
                count,
                reference_ngrams[ngram],
            )
            for ngram, count
            in predicted_ngrams.items()
        )

        total = sum(
            predicted_ngrams.values()
        )

        precisions.append(
            (clipped + 1.0)
            / (total + 1.0)
        )

    reference_length = len(
        reference_tokens
    )
    prediction_length = len(
        prediction_tokens
    )

    brevity_penalty = (
        1.0
        if prediction_length
        > reference_length
        else math.exp(
            1.0
            - reference_length
            / max(
                prediction_length,
                1,
            )
        )
    )

    geometric_mean = math.exp(
        sum(
            math.log(
                max(
                    precision,
                    1e-12,
                )
            )
            for precision in precisions
        )
        / maximum_order
    )

    return (
        brevity_penalty
        * geometric_mean
    )


test_results[
    "bleu_like"
] = [
    sentence_bleu_like(
        reference,
        prediction,
    )
    for reference, prediction in zip(
        test_results["reference"],
        test_results["prediction"],
    )
]

print(
    "Mean BLEU-like score:",
    round(
        test_results[
            "bleu_like"
        ].mean(),
        3,
    ),
)

Formal translation studies should use established metric implementations and
corpus-level reporting.

# 29. Error Analysis

In [ ]:
test_results.sort_values(
    [
        "exact_match",
        "bleu_like",
        "mean_token_confidence",
    ],
    ascending=[
        True,
        True,
        False,
    ],
).reset_index(drop=True)

In [ ]:
error_categories = []

for row in test_results.itertuples(
    index=False
):
    reference_tokens = tokenize(
        row.reference
    )
    prediction_tokens = tokenize(
        row.prediction
    )

    if row.exact_match:
        category = "correct"
    elif len(prediction_tokens) < len(
        reference_tokens
    ):
        category = "too short"
    elif len(prediction_tokens) > len(
        reference_tokens
    ):
        category = "too long"
    elif (
        set(prediction_tokens)
        == set(reference_tokens)
    ):
        category = "word order"
    else:
        category = "lexical or agreement error"

    error_categories.append(category)

test_results[
    "error_category"
] = error_categories

test_results[
    "error_category"
].value_counts()

# 30. Exposure Bias

During teacher-forced training, the decoder often sees correct histories.
During inference, it sees its own predictions.

One early error changes all later decoder inputs.

In [ ]:
exposure_bias = pd.DataFrame(
    [
        (
            "Training",
            "mostly gold prefixes",
            "clean context",
        ),
        (
            "Inference",
            "predicted prefixes",
            "errors can accumulate",
        ),
    ],
    columns=["Phase", "Prefix source", "Consequence"],
)

exposure_bias

Scheduled teacher forcing reduces but does not eliminate this mismatch.

# 31. Decoding Length

Autoregressive decoding stops when:

- EOS is generated;
- a maximum length is reached.

In [ ]:
test_results[
    "reference_length"
] = test_results[
    "reference"
].map(
    lambda text: len(tokenize(text))
)

test_results[
    "prediction_length"
] = test_results[
    "prediction"
].map(
    lambda text: len(tokenize(text))
)

test_results[
    [
        "reference_length",
        "prediction_length",
    ]
].describe()

A maximum length prevents infinite generation but can truncate valid outputs.

# 32. Repetition and Premature EOS

Common decoder failures include:

- repeated words;
- missing modifiers;
- early EOS;
- failure to emit EOS;
- agreement errors.

In [ ]:
def has_repetition(text: str) -> bool:
    tokens = tokenize(text)

    return any(
        tokens[index]
        == tokens[index - 1]
        for index in range(
            1,
            len(tokens),
        )
    )


test_results[
    "immediate_repetition"
] = test_results[
    "prediction"
].map(
    has_repetition
)

test_results[
    "immediate_repetition"
].value_counts()

# 33. Beam Search Concept

Beam search keeps several partial hypotheses at each step.

```text
greedy search: retain 1 hypothesis
beam search:   retain k hypotheses
```

In [ ]:
search_comparison = pd.DataFrame(
    [
        ("Greedy", 1, "fast", "locally optimal"),
        ("Beam", "k", "slower", "broader search"),
        ("Sampling", "variable", "diverse", "stochastic"),
    ],
    columns=["Method", "Active hypotheses", "Speed", "Property"],
)

search_comparison

A wider beam does not guarantee better human-quality output and can favor
short sequences without appropriate scoring.

# 34. Fixed-Vector Bottleneck

In this architecture, the decoder receives only the encoder's final hidden
state.

For long sequences, one vector must preserve all relevant source information.

In [ ]:
bottleneck_effects = pd.DataFrame(
    [
        ("Long source", "more information compressed"),
        ("Complex reordering", "alignment is implicit"),
        ("Rare details", "may disappear from context"),
        ("Long dependencies", "decoder receives limited access"),
    ],
    columns=["Condition", "Potential effect"],
)

bottleneck_effects

Attention allows the decoder to consult all encoder states instead of relying
on one fixed vector.

# 35. Bidirectional Encoders

A bidirectional encoder reads the complete source in both directions.

Benefits:

- left and right source context;
- stronger source representation.

Costs:

- more parameters;
- larger context state;
- unsuitable for strictly streaming source input.

The decoder remains autoregressive even when the encoder is bidirectional.

# 36. Multi-Layer Encoder–Decoders

Stacking recurrent layers increases representational capacity.

Design decisions include:

- matching encoder and decoder layer counts;
- projecting encoder states;
- recurrent dropout;
- residual connections;
- hidden-size compatibility.

In [ ]:
state_compatibility = pd.DataFrame(
    [
        ("Same hidden size", "direct state transfer"),
        ("Different hidden size", "learned projection"),
        ("Bidirectional encoder", "combine or project directions"),
        ("Different layer counts", "select, repeat, or transform states"),
    ],
    columns=["Configuration", "Required handling"],
)

state_compatibility

# 37. Regularization and Stability

Useful techniques:

- dropout;
- weight decay;
- gradient clipping;
- early stopping;
- teacher-forcing schedules;
- label smoothing;
- larger and cleaner data.

In [ ]:
stability_summary = pd.DataFrame(
    [
        ("Gradient clipping", "exploding gradients"),
        ("Early stopping", "overfitting"),
        ("Dropout", "co-adaptation"),
        ("Teacher schedule", "train–inference mismatch"),
        ("Label smoothing", "overconfidence"),
    ],
    columns=["Technique", "Primary target"],
)

stability_summary

# 38. Computational Cost

Encoder recurrence is sequential across source positions. Decoder recurrence is
sequential across generated positions.

In [ ]:
cost_factors = pd.DataFrame(
    [
        ("Source length", "encoder steps"),
        ("Target length", "decoder steps"),
        ("Vocabulary size", "output projection and softmax"),
        ("Beam width", "number of active hypotheses"),
        ("Hidden dimension", "recurrent matrix cost"),
    ],
    columns=["Factor", "Cost contribution"],
)

cost_factors

Large target vocabularies can make the output layer one of the model's largest
components.

# 39. Arabic and Multilingual Considerations

Arabic sequence-to-sequence systems must account for:

- rich morphology;
- clitic attachment;
- optional tashkeel;
- orthographic variation;
- MSA and dialects;
- word-order differences;
- agreement;
- code-switching.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "وَسَيَكْتُبُونَهَا",
            "وَ + سَ + يَكْتُبُونَ + هَا",
        ),
        (
            "بِالْمَدْرَسَةِ",
            "بِ + الْمَدْرَسَةِ",
        ),
        (
            "كِتَابُهُمَا",
            "كِتَابُ + هُمَا",
        ),
    ],
    columns=[
        "Fully vocalized form",
        "Illustrative segmentation",
    ],
)

arabic_examples

Word-level vocabularies may fragment Arabic coverage across many inflected
forms. Subword and morphological segmentation can reduce sparsity.

In [ ]:
multilingual_design = pd.DataFrame(
    [
        ("Separate vocabularies", "clear language separation"),
        ("Shared subword vocabulary", "cross-lingual sharing"),
        ("Language tags", "control target language"),
        ("Tashkeel preservation", "retain vocalized distinctions"),
        ("Dialect labels", "represent language variety"),
    ],
    columns=["Design choice", "Purpose"],
)

multilingual_design

For fully vocalized Arabic tasks, removing diacritics changes the supervised
target and should not be treated as harmless normalization.

# 40. Reproducibility and Reporting

Report:

- corpus and split;
- source and target preprocessing;
- vocabularies;
- special tokens;
- embedding and hidden dimensions;
- encoder and decoder cell types;
- teacher-forcing schedule;
- optimizer and learning rate;
- batch size;
- clipping and early stopping;
- decoding method;
- maximum generation length;
- evaluation metrics;
- random seeds;
- hardware.

In [ ]:
import platform

metadata = pd.Series(
    {
        "sentence_pairs": len(dataset),
        "training_pairs": len(train_frame),
        "validation_pairs": len(validation_frame),
        "test_pairs": len(test_frame),
        "source_vocabulary": len(source_vocabulary),
        "target_vocabulary": len(target_vocabulary),
        "embedding_dimension": 32,
        "hidden_dimension": 48,
        "encoder": "GRU",
        "decoder": "GRU",
        "decoding": "greedy",
        "device": str(DEVICE),
        "random_seed": 42,
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
    },
    name="Seq2Seq experiment",
)

metadata

# 41. Knowledge Check

1. What is a sequence-to-sequence task?
2. What does the encoder produce?
3. Why does the decoder need BOS?
4. What does EOS control?
5. Why are source and target vocabularies often separate?
6. What is autoregressive decoding?
7. What is teacher forcing?
8. Why must padding be ignored in sequence loss?
9. How do training and inference differ?
10. What is exposure bias?
11. How does greedy decoding work?
12. Why is exact match strict?
13. What does beam search retain?
14. What is the fixed-vector bottleneck?
15. Why does attention help?
16. Which Arabic properties affect sequence-to-sequence learning?

# 42. Exercises

## Exercise 1 — Vocabulary

Add minimum-frequency filtering and report OOV rates.

## Exercise 2 — Teacher Forcing

Compare constant, linear, and exponential schedules.

## Exercise 3 — Decoder Depth

Compare one-layer and two-layer decoders.

## Exercise 4 — Cell Type

Replace GRU cells with LSTM cells.

## Exercise 5 — Bidirectional Encoder

Implement a bidirectional encoder and project its state.

## Exercise 6 — Beam Search

Implement beam search with length normalization.

## Exercise 7 — Evaluation

Add corpus BLEU and chrF using established libraries.

## Exercise 8 — Robustness

Test unseen word combinations and noisy source text.

## Exercise 9 — Arabic Translation

Build a small fully vocalized Arabic translation task.

## Exercise 10 — Error Analysis

Categorize lexical, agreement, ordering, and length errors.

## Challenge Exercises

1. Add attention over encoder states.
2. Visualize source–target alignment weights.
3. Implement scheduled sampling per sequence position.
4. Add label smoothing.
5. Build a multilingual decoder controlled by language tags.

# 43. Summary and Next Lesson

In this lesson:

- sequence-to-sequence learning mapped source sequences to target sequences;
- encoder and decoder roles were separated;
- PAD, UNK, BOS, and EOS controlled batching and generation;
- separate source and target vocabularies were constructed;
- recurrent source encoding produced a context state;
- autoregressive decoding generated target tokens;
- teacher forcing supported supervised sequence training;
- masked cross-entropy ignored padded targets;
- a CPU-only GRU encoder–decoder was trained with PyTorch;
- greedy decoding produced complete target sequences;
- exact match, token accuracy, and BLEU-like evaluation were implemented;
- exposure bias and decoding errors were analyzed;
- the fixed-vector bottleneck motivated dynamic source access;
- Arabic morphology, clitics, agreement, and tashkeel were connected to
  sequence-to-sequence design.

## Next Lesson

**Lesson 30: Attention Mechanisms for Neural Sequence Models** introduces
alignment scores, attention distributions, context vectors, masked attention,
attention visualization, and attentive encoder–decoder translation.

# References

- Sutskever, I., Vinyals, O., & Le, Q. V. sequence-to-sequence learning.
- Cho, K. et al. recurrent encoder–decoder learning.
- Bengio, S. et al. scheduled sampling.
- Goodfellow, I., Bengio, Y., & Courville, A. *Deep Learning*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.